# 2017 SOXX 修订申报来源

四份精确 URL、原始版和时点边界见 `plan.json`。

In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_2017_amendment_semantics_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['original_numeric_result_sha256']==sha(root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023/numeric_2018_Jun_dollar_cell_repair_result.json')
assert plan['frozen'] and plan['series_calendar_sha256']==sha(root/'research/stage1_soxx_series_filing_search_20261023/series_report_calendar_inventory.json')
rawdir=root/'data/snapshots/stage1_soxx_2017_amendment_semantics_20261023';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'attempt_started.json';out=card/'transport_result.json'
assert not claim.exists() and not out.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'maximum_GETs':4},indent=2)+'\n')
requests=[]
for j,item in enumerate(plan['reports']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 raw=rawdir/(item['accession']+'_'+item['primary_document']);headers=rawdir/(raw.name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: text/html',item['url']]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 entry={'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'url':item['url'],'requested_at_utc':started,'received_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
  'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 requests.append(entry)
 if proc.returncode or status!=200 or entry['final_url']!=item['url'] or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  entry['stop_reason']='transport_or_identity_failed';break
receipt={'status':'four_original_2017_amendment_bodies_received_semantics_pending' if len(requests)==4 and all('stop_reason' not in x for x in requests) else '2017_amendment_body_batch_stopped',
 'plan_sha256':sha(card/'plan.json'),'GETs_consumed':len(requests),'requests':requests,'SOXX_schedule_identities_qualified':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2017-amendment-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,out,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2017-amendment-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out_file:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out_file,stderr=subprocess.PIPE,timeout=120)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for entry in requests:
    if entry['raw_sha256'] is None:continue
    member=stored.extractfile(Path(entry['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==entry['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(requests),'raw_MB':round(sum(x['raw_bytes'] for x in requests)/1e6,2),'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'four_original_2017_amendment_bodies_received_semantics_pending', 'GETs': 4, 'raw_MB': 227.08, 'NAS_restore': True}


In [1]:
import json,hashlib,re,copy
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2017_amendment_semantics_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
transport=json.loads((card/'transport_result.json').read_text());assert transport['status']=='four_original_2017_amendment_bodies_received_semantics_pending' and transport['nas_scoped_restore_verified']
assert transport['plan_sha256']==sha(card/'plan.json') and plan['original_numeric_result_sha256']==sha(root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023/numeric_2018_Jun_dollar_cell_repair_result.json')
original=json.loads((root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023/numeric_2018_Jun_dollar_cell_repair_result.json').read_text())
base={x['report_date']:x for x in original['reports'] if x['report_date'] in ('2017-03-31','2017-09-30')}
assert len(base)==2 and not (card/'semantics_result.json').exists()
claim=card/'semantics_attempt_started.json';assert not claim.exists();claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'raw_amendment_checks_started':4},indent=2)+'\n')
integer=re.compile(r'^[0-9][0-9,]*$');money=re.compile(r'^\$\s*([0-9][0-9,]*)$')
results=[];previous={k:copy.deepcopy(v['equities']) for k,v in base.items()};uncertain={k:False for k in base}
target=plan['target_fund_title'].upper()
for source in transport['requests']:
 raw=root/source['raw_path'];body=raw.read_bytes();assert hashlib.sha256(body).hexdigest()==source['raw_sha256']
 expected=datetime.strptime(source['report_date'],'%Y-%m-%d').strftime('%B %d, %Y')
 locations=[];selected=None
 for match in re.finditer(rb'<A\s+HREF="#([^"]+)"[^>]*>(.*?)</A>',body,re.I|re.S):
  label=' '.join(BeautifulSoup(match.group(2).decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split()).upper()
  if target not in label:continue
  anchor=match.group(1).decode('ascii',errors='ignore')
  found=re.search(rb'NAME="'+re.escape(anchor.encode())+rb'"',body[match.end():],re.I)
  if found is None:continue
  start=match.end()+found.start();next_match=re.search(rb'NAME="toc[^"]+"',body[start+10:],re.I)
  end=(start+10+next_match.start()) if next_match else min(len(body),start+100000)
  section=body[start:min(end,start+75000)]
  plain=' '.join(BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
  passed=target in plain.upper() and expected in plain and all(x in plain for x in ('Security','Shares','Value')) and 'COMMON STOCKS' in plain.upper()
  locations.append({'method':'fund_named_TOC_after_link','anchor':anchor,'fund_link_offset':match.start(),'start':start,'end':min(end,start+75000),'section_sha256':hashlib.sha256(section).hexdigest(),'identity_pass':passed})
  if passed:selected=(locations[-1],section);break
 if selected is None:
  for match in re.finditer(re.escape(target.lower().encode()),body.lower()):
   start=match.start();section=body[start:min(len(body),start+75000)]
   plain=' '.join(BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
   passed=expected in plain[:1800] and all(x in plain[:3500] for x in ('Security','Shares','Value')) and 'COMMON STOCKS' in plain.upper()
   locations.append({'method':'bounded_inline_title','start':start,'end':min(len(body),start+75000),'section_sha256':hashlib.sha256(section).hexdigest(),'identity_pass':passed})
   if passed:selected=(locations[-1],section);break
 row={'report_date':source['report_date'],'form':source['form'],'accession':source['accession'],'accepted_at_utc':source['accepted_at_utc'],
  'raw_sha256':source['raw_sha256'],'raw_bytes':source['raw_bytes'],'identity_locations':locations,
  'SOXX_equity_schedule_identity_pass':selected is not None,'status':'SOXX_schedule_identity_unqualified'}
 if selected is not None:
  where,section=selected;parsed=[];ambiguous=[];started=False;total_label=False;printed=None
  for html_row in BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').find_all('tr'):
   cells=[' '.join(x.get_text(' ',strip=True).replace('\xa0',' ').split()) for x in html_row.find_all(['td','th'],recursive=False)]
   nonempty=[x for x in cells if x and x!='$']
   if not nonempty:continue
   name=nonempty[0];upper=name.upper();nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
   money_values=[int(m.group(1).replace(',','')) for x in nonempty[1:] if (m:=money.fullmatch(x))]
   if not started:
    if 'COMMON STOCKS' in upper and 'TOTAL' not in upper:started=True
    continue
   if 'TOTAL COMMON STOCKS' in upper:
    total_label=True
    if nums or money_values:printed=(money_values or nums)[-1];break
    continue
   if total_label:
    if nums or money_values:printed=(money_values or nums)[-1]
    break
   if upper==name and ('—' in name or '–' in name or 'COMMON STOCKS' in upper):continue
   if nums and (len(nums)>=2 or money_values) and any(ch.isalpha() for ch in name):
    parsed.append({'security_original':name,'shares_original':nums[0],'fair_value_usd_original':(money_values or nums[1:])[-1]})
   elif any(ch.isalpha() for ch in name) and (nums or money_values):
    ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums,'money_tokens':money_values})
  values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
  for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
  checks={'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),
   'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),'printed_total_found':printed is not None,
   'total_closure_within_1_USD':printed is not None and abs(total-printed)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
  row.update(selected_location=where,parsed_equity_rows=len(parsed),equities=parsed,ambiguous_rows=ambiguous,
   printed_total_common_stocks_usd=printed,parsed_fair_value_total_usd=total,numeric_checks=checks,
   top5_disclosed_equity_value_share=sum(sorted((x['reported_equity_weight'] for x in parsed),reverse=True)[:5]) if total else None,
   HHI=sum(x['reported_equity_weight']**2 for x in parsed) if total else None)
  if all(checks.values()):
   key=lambda rr:sorted((x['security_original'],x['shares_original'],x['fair_value_usd_original']) for x in rr)
   base_row=base[source['report_date']]
   row['exact_equity_rows_equal_original']=key(parsed)==key(base_row['equities'])
   row['original_total_USD_delta']=total-base_row['parsed_fair_value_total_usd']
   row['original_top5_delta']=row['top5_disclosed_equity_value_share']-base_row['top5_disclosed_equity_value_share']
   row['original_HHI_delta']=row['HHI']-base_row['herfindahl_sum_squared_disclosed_equity_weights']
   row['previous_qualified_vintage_comparison_unbroken']=not uncertain[source['report_date']]
   if not uncertain[source['report_date']]:row['exact_equity_rows_equal_previous_qualified_vintage']=key(parsed)==key(previous[source['report_date']])
   previous[source['report_date']]=copy.deepcopy(parsed)
   row['status']='SOXX_amended_equity_numeric_qualified_'+('unchanged_from_original' if row['exact_equity_rows_equal_original'] else 'changed_from_original')
  else:row['status']='SOXX_amended_equity_numeric_unqualified'
 if not row['status'].startswith('SOXX_amended_equity_numeric_qualified'):uncertain[source['report_date']]=True
 results.append(row)
summary={'status':'all_four_2017_amendment_SOXX_equity_semantics_qualified' if all(x['status'].startswith('SOXX_amended_equity_numeric_qualified') for x in results) else '2017_amendment_SOXX_equity_semantics_partial_or_unqualified',
 'plan_sha256':sha(card/'plan.json'),'amendments':results,'amendment_GETs_completed':transport['GETs_consumed'],
 'numeric_qualified_count':sum(x['status'].startswith('SOXX_amended_equity_numeric_qualified') for x in results),
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'original_source_result_sha256':plan['original_numeric_result_sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'availability_boundary':'No amended value may be used before its own accepted-at time and a later actual Qlib session; preserve every original source version.'}
(card/'semantics_result.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print({'status':summary['status'],'qualified':summary['numeric_qualified_count'],'amendments':[(x['accession'],x['status']) for x in results]})


{'status': 'all_four_2017_amendment_SOXX_equity_semantics_qualified', 'qualified': 4, 'amendments': [('0001193125-17-257387', 'SOXX_amended_equity_numeric_qualified_changed_from_original'), ('0001193125-18-009766', 'SOXX_amended_equity_numeric_qualified_changed_from_original'), ('0001193125-18-152961', 'SOXX_amended_equity_numeric_qualified_changed_from_original'), ('0001193125-18-152996', 'SOXX_amended_equity_numeric_qualified_changed_from_original')]}


## 中文结论

执行后见 `summary.json` 和 `conclusion.md`；修订来源不等于 SOXX 超额。